# Open Insect ID v2 — Export to ONNX / INT8 / TFLite

CPU notebook. Loads the trained MoE, exports every router + expert to ONNX,
applies INT8 post-training quantization, and converts to `.tflite` for the
Android app. Attach the checkpoint output-dataset from notebook 02.


In [ ]:
import os, subprocess, sys
from pathlib import Path

# --- Configure this to your repo -------------------------------------------
GITHUB_REPO = os.environ.get("OPENINSECT_REPO", "https://github.com/YoannDev90/OpenInsectID.git")
# ---------------------------------------------------------------------------
IS_KAGGLE = os.path.exists("/kaggle/input")

def _is_our_checkout(path):
    # Identify OUR repo precisely: any folder happening to contain a src/
    # (cwd leftovers, /kaggle, ...) must not be mistaken for the checkout.
    return (os.path.isfile(os.path.join(path, "pyproject.toml"))
            and os.path.isdir(os.path.join(path, "src", "data_pipeline")))

# Walk up from cwd (kernels start in /kaggle/working, Jupyter in the notebook
# folder) to the repo root; anywhere else means: install from GitHub.
repo_root = os.getcwd()
while not _is_our_checkout(repo_root):
    parent = os.path.dirname(repo_root)
    if parent == repo_root:
        break
    repo_root = parent
in_checkout = _is_our_checkout(repo_root)

if in_checkout:
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
else:
    # pip needs the `git+` prefix to clone -- a plain https URL makes GitHub
    # serve the HTML page and pip fails with "cannot detect archive format".
    install_url = GITHUB_REPO
    if install_url.startswith(("http://", "https://")) and not install_url.startswith("git+"):
        install_url = "git+" + install_url
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", install_url])

try:
    import src.data_pipeline  # noqa: F401 - fail fast, with context, if shadowed
except ModuleNotFoundError:
    print("DIAG sys.path:", sys.path)
    raise

from pathlib import Path
import yaml

print(f"cwd={os.getcwd()} checkout={in_checkout} repo_root={repo_root}")
print("package ready; importable as `src.*`")


## Load the trained model

In [ ]:
import torch
from src.models import HierarchicalMoE, TaxonomyTree

CKPT = "/kaggle/input/open-insect-id-checkpoints/moe_state.pt"
TAX  = "/kaggle/input/open-insect-id-checkpoints/taxonomy.json"
if not Path(CKPT).exists():
    CKPT, TAX = "checkpoints/moe_state.pt", "checkpoints/taxonomy.json"

tree = TaxonomyTree.load(TAX)
model = HierarchicalMoE(tree, backbone_name="mnv4_small")
model.load_state_dict(torch.load(CKPT, map_location="cpu"))
model.eval()
print("loaded MoE:", model.n_orders, "orders /", model.n_species, "species")


## Export per-module ONNX

In [ ]:
from src.export import export_moe
from pathlib import Path

onnx_dir = Path("export/onnx"); onnx_dir.mkdir(parents=True, exist_ok=True)
paths = export_moe(model, onnx_dir, image_size=224)
print(f"exported {len(paths)} modules -> {onnx_dir}")
print("manifest:", (onnx_dir / "manifest.json").exists())


## INT8 post-training quantization (with calibration set)

In [ ]:
from src.export import quantize_static_int8, quantize_dynamic_int8
import torch

# Calibration images: a representative batch from your validation set.
calib = torch.randn(64, 3, 224, 224)

qdir = Path("export/onnx_int8"); qdir.mkdir(parents=True, exist_ok=True)
router_onnx = onnx_dir / "router.onnx"
if router_onnx.exists():
    q = quantize_static_int8(router_onnx, qdir / "router_int8.onnx", calib)
    print("quantized router ->", q)
else:
    print("router.onnx not found; run the export cell first.")


## Convert to TFLite

Uses `onnx2tf` when available. Install it with
`pip install onnx2tf onnx==1.17.0 tflite-support` on the Kaggle kernel
(via `!pip install ...`). If it is absent the converter prints the exact
command to run instead of failing.


In [ ]:
from src.export import TFLiteINT8Converter

conv = TFLiteINT8Converter()
tflite_dir = Path("export/tflite"); tflite_dir.mkdir(parents=True, exist_ok=True)
if conv.available() and router_onnx.exists():
    tfl = conv.convert(router_onnx, tflite_dir, int8=True)
    print("tflite ->", tfl)
else:
    print("onnx2tf not installed. Install with:")
    print("  !pip install onnx2tf onnx==1.17.0 tflite-support")
    print("then re-run this cell.")


## Package the artifact

Bundle `export/tflite/`, `manifest.json`, and `taxonomy.json` into a Kaggle
dataset (or zip for GitHub Releases). The Android app loads the router first,
then fetches only the experts along the predicted taxonomic path (Option A).


In [ ]:
import shutil, json
bundle = Path("export/open_insect_id_bundle")
bundle.mkdir(parents=True, exist_ok=True)
for src in [Path("export/tflite"), onnx_dir / "manifest.json"]:
    if src.exists():
        if src.is_dir():
            shutil.copytree(src, bundle / src.name, dirs_exist_ok=True)
        else:
            shutil.copy(src, bundle / src.name)
shutil.copy(TAX, bundle / "taxonomy.json")
shutil.make_archive("open_insect_id_tflite", "zip", root_dir=bundle)
print("bundle:", list(p.name for p in bundle.iterdir()))
